In [ ]:
%pip install -q pandas transformers torch tqdm

Lyrics Emotion Classification Pipeline
=======================================
Input:  analysis_df with columns: rank, artist, title, region,
        spotify_uri, lyrics_in_en, original_lang
Output: analysis_df + 13 emotion score columns + dominant_emotion column

In [ ]:
import re
import pandas as pd
from tqdm import tqdm
from transformers import pipeline
from pathlib import Path
from datetime import datetime

DATE_PATH = '2026/03/05'
PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root

analysis_df = pd.read_csv(PROJECT_ROOT / "data/processed/lyrics/" / DATE_PATH / "lyrics_in_en.csv")
# analysis_df = pd.read_csv("/Volumes/songs_db/default/storage/lyrics_in_en.csv")

print(f'Loaded {len(analysis_df)} songs')
print(f'Columns: {list(analysis_df.columns)}')

display(analysis_df.head(3))

In [ ]:
# Config

EMOTIONS = [
    "love", "angst", "joy", "heartbreak", "empowerment",
    "party", "despair", "hope", "nostalgia", "solitude",
    "passion", "relaxing", "other"
]
 
# How many words per chunk (stay under ~400 tokens)
CHUNK_WORDS = 350
 
# Score threshold: below this, treat as 0 (noise floor for multi-label)
NOISE_FLOOR = 0.10

### Clean lyrics

In [ ]:
def clean_lyrics(text: str, artist: str = '') -> str:
    """
    Remove structural noise but preserve punctuation and casing.
    Punctuation (! ? ...) and capitalisation carry emotional signal
    for NLI-based classifiers — don't strip them.
 
    artist: the artist string from the dataframe row. When provided,
    the first line is dropped only if its tokens are a subset of the
    known artist names — much more precise than a regex heuristic.
    Falls back to the regex heuristic when artist is unavailable.
    """
    if not isinstance(text, str) or not text.strip():
        return ''
 
    def is_artist_credit(line: str) -> bool:
        """
        True if every name token in `line` exists in the artist pool.
        Both strings are split on commas, ampersands, and feat/ft.
 
            artist = "Jason, Bonnie"          line = "Jason"          → True
            artist = "ARIA VEGA, Ryan Castro" line = "Ryan Castro"    → True
            artist = "Jason, Bonnie"          line = "Baby come back" → False
        """
        splitter = r'[,&]|\bfeat\.?\b|\bft\.?\b'
        artist_tokens = {
            t.strip().lower()
            for t in re.split(splitter, artist, flags=re.IGNORECASE)
            if t.strip()
        }
        line_tokens = {
            t.strip().lower()
            for t in re.split(splitter, line, flags=re.IGNORECASE)
            if t.strip()
        }
        return bool(line_tokens) and line_tokens.issubset(artist_tokens)
 
    # Remove section headers: [Verse 1], [Chorus], [Bridge] etc.
    text = re.sub(r'\[[^\]]*\]', '', text)
 
    # Remove repetition annotations: (x3), (×2), (2x)
    text = re.sub(r'\([\d×xX]+\)', '', text)
 
    # Remove leading artist/feature credits that sometimes appear
    # at the top of translated lyrics (e.g. "ARIA VEGA, Ryan Castro\n").
    lines = text.strip().splitlines()
    if lines:
        first = lines[0].strip()
        if artist and is_artist_credit(first):
            lines = lines[1:]
        elif not artist and re.match(r'^[A-Za-z\s,&]+$', first) and len(first) < 80:
            lines = lines[1:]
    text = '\n'.join(lines)
 
    # Collapse excessive blank lines but keep single line breaks
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
 
    return text.strip()

def dedupe_lines(text: str) -> str:
    """
    Remove duplicate lines (repeated choruses inflate scores).
    Keeps first occurrence; preserves order.
    """
    seen = set()
    result = []
    for line in text.splitlines():
        key = line.strip().lower()
        if key and key not in seen:
            seen.add(key)
            result.append(line.strip())
    return ' '.join(result)
 
 
def prepare_lyrics(text: str, artist: str = '') -> str:
    return dedupe_lines(clean_lyrics(text, artist=artist))

### Chunking & Classification

In [ ]:
# ─────────────────────────────────────────────
# Chunking long lyrics for NLI classification
# ─────────────────────────────────────────────

def chunk_text(text: str, chunk_words: int = CHUNK_WORDS) -> list[str]:
    """Split text into word-count chunks with a small overlap."""
    words = text.split()
    if not words:
        return []
    overlap = 30
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_words
        chunks.append(' '.join(words[start:end]))
        start = end - overlap  # small overlap so context isn't lost at boundaries
        if start >= len(words):
            break
    return chunks
 
 
# ─────────────────────────────────────────────
# Classification
# ─────────────────────────────────────────────
 
def load_classifier():
    """
    facebook/bart-large-mnli  — best general zero-shot NLI classifier.
    Falls back to a smaller model if memory is tight.
    Switch to cross-encoder/nli-deberta-v3-large for higher accuracy
    at the cost of ~2× slower inference.
    """
    print("Loading zero-shot classifier (bart-large-mnli)...")
    return pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=0  # set to -1 to force CPU; 0 for first GPU
    )
 
 
def classify_all_songs(lyrics_series: pd.Series, classifier) -> list[dict]:
    """
    Classify all songs in a single batched classifier call.
 
    Instead of looping chunk-by-chunk per song, we:
      1. Chunk every song and flatten into one big list of chunks
      2. Record which song each chunk belongs to (chunk_map)
      3. Send the entire list to the classifier in one call with batch_size
      4. Reassemble and average scores back per song
 
    This lets the model fill its batch queue continuously — no idle time
    between songs — which is the primary speedup on both GPU and CPU.
 
    batch_size: tune this to your GPU VRAM. 8 is safe for ~8GB VRAM.
    Reduce to 4 if you get OOM errors; raise to 16+ on larger GPUs.
    """
    BATCH_SIZE = 8
 
    # Step 1: build flat chunk list and a map back to song index
    all_chunks = []   # flat list of chunk strings
    chunk_map = []    # parallel list: which song index does this chunk belong to
 
    for song_idx, lyrics in lyrics_series.items():
        if not isinstance(lyrics, str) or not lyrics.strip():
            continue
        chunks = chunk_text(lyrics)
        all_chunks.extend(chunks)
        chunk_map.extend([song_idx] * len(chunks))
 
    if not all_chunks:
        return [{e: 0.0 for e in EMOTIONS}] * len(lyrics_series)
 
    # Step 2: manually batch all chunks and call the classifier per batch.
    # This is the only reliable way to get a tqdm bar that ticks steadily —
    # passing a generator to the pipeline doesn't work because HuggingFace
    # buffers the entire generator internally before starting inference,
    # so the bar would jump from 0% to 100% at the end regardless.
    n_batches = -(-len(all_chunks) // BATCH_SIZE)  # ceiling division
    print(f"  Classifying {len(all_chunks)} chunks "
          f"({len(lyrics_series)} songs) in {n_batches} batches...")
 
    raw_results = []
    for i in tqdm(range(0, len(all_chunks), BATCH_SIZE),
                  total=n_batches,
                  desc="Classifying batches",
                  unit="batch"):
        batch = all_chunks[i : i + BATCH_SIZE]
        raw_results.extend(
            classifier(batch, candidate_labels=EMOTIONS, multi_label=True)
        )
 
    # Step 3: group chunk scores back by song index
    song_chunks: dict[int, list[dict]] = {}
    for chunk_result, song_idx in zip(raw_results, chunk_map):
        scores = dict(zip(chunk_result["labels"], chunk_result["scores"]))
        song_chunks.setdefault(song_idx, []).append(scores)
 
    # Step 4: average scores per song and apply noise floor
    def average_and_floor(score_list: list[dict]) -> dict:
        avg = {
            e: round(sum(s[e] for s in score_list) / len(score_list), 4)
            for e in EMOTIONS
        }
        return {e: (v if v >= NOISE_FLOOR else 0.0) for e, v in avg.items()}
 
    return [
        average_and_floor(song_chunks[idx]) if idx in song_chunks
        else {e: 0.0 for e in EMOTIONS}
        for idx in lyrics_series.index
    ]

In [ ]:
df = analysis_df.copy()

 # Step 1: Prepare lyrics
print("Cleaning and deduplicating lyrics...")
df['lyrics_prepared'] = df.apply(
    lambda row: prepare_lyrics(
        row['lyrics_in_en'] if isinstance(row['lyrics_in_en'], str) else '',
        artist=row['artist'] if isinstance(row['artist'], str) else ''
    ),
    axis=1
)

has_lyrics = df['lyrics_prepared'].str.strip().ne('')
print(f"Songs with usable lyrics: {has_lyrics.sum()} / {len(df)}")


display(df)

In [ ]:
# Step 2: Load model
classifier = load_classifier()

# Step 3: Classify all songs in one batched call
print(f"\nClassifying {has_lyrics.sum()} songs across emotions...")
results = classify_all_songs(df['lyrics_prepared'], classifier)

# Step 4: Merge scores into dataframe
scores_df = pd.DataFrame(results, index=df.index)
scores_df.columns = [f"emotion_{e}" for e in EMOTIONS]
df = pd.concat([df, scores_df], axis=1)

# Step 5: Derive dominant emotion per song
emotion_cols = [f"emotion_{e}" for e in EMOTIONS]
df['dominant_emotion'] = df[emotion_cols].idxmax(axis=1).str.replace('emotion_', '', regex=False)
df['dominant_emotion'] = df['dominant_emotion'].where(
    df[emotion_cols].max(axis=1) > NOISE_FLOOR, other='unclassified'
)

# Drop working column
df = df.drop(columns=['lyrics_prepared'])

df.to_csv(PROJECT_ROOT / "data/processed/lyrics/" / DATE_PATH / "lyrics_analysis.csv", index=False)
# df.to_csv("/Volumes/songs_db/default/storage/lyrics_analysis.csv", index=False)

In [ ]:
def regional_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Average emotion scores per region.
    Useful for radar/heatmap visualisation.
    """
    emotion_cols = [f"emotion_{e}" for e in EMOTIONS]
    summary = df.groupby('region')[emotion_cols].mean().round(3)
    summary.columns = [c.replace('emotion_', '') for c in summary.columns]
    return summary

summary = regional_summary(df)
display(summary)
summary.to_csv(PROJECT_ROOT / "data/processed/lyrics/" / DATE_PATH / "regional_summary.csv", index=False)
# summary.to_csv("/Volumes/songs_db/default/storage/regional_summary.csv", index=False)
